In [47]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
from datetime import datetime

# --- Helper functions ---
def get_title(soup):
    tag = soup.find("span", id="productTitle")
    return tag.get_text(strip=True) if tag else ""

def get_price(soup):
    # Try standard IDs first
    for pid in ["priceblock_ourprice", "priceblock_dealprice", "priceblock_saleprice"]:
        tag = soup.find("span", id=pid)
        if tag:
            return tag.get_text(strip=True)

    # Fallback: new price format
    whole = soup.select_one("span.a-price-whole")
    frac = soup.select_one("span.a-price-fraction")
    if whole:
        return f"${whole.get_text(strip=True)}{frac.get_text(strip=True) if frac else ''}"

    return ""

def get_rating(soup):
    tag = soup.select_one("span.a-icon-alt")
    return tag.get_text(strip=True) if tag else ""

def get_review_count(soup):
    tag = soup.find("span", id="acrCustomerReviewText")
    return tag.get_text(strip=True) if tag else ""

def get_availability(soup):
    tag = soup.select_one("div#availability span")
    return tag.get_text(strip=True) if tag else "Not Available"

def get_brand(soup):
    tag = soup.find("a", id="bylineInfo")
    return tag.get_text(strip=True) if tag else ""

def get_category(soup):
    tag = soup.select_one("#wayfinding-breadcrumbs_feature_div ul li span")
    return tag.get_text(strip=True) if tag else ""

def get_base_type(title):
    title_lower = title.lower()
    if "oil based" in title_lower or "oil-based" in title_lower:
        return "Oil-Based"
    elif "water based" in title_lower or "water-based" in title_lower:
        return "Water-Based"
    else:
        return "Unknown"

# --- Safe converters ---
def safe_float(value):
    try:
        return float(value.replace("$","").replace(",",""))
    except:
        return None

def safe_int(value):
    try:
        # Remove parentheses and then commas
        cleaned_value = value.strip('()').replace(",","")
        return int(cleaned_value.split()[0])
    except:
        return None

# --- Main scraper ---
if __name__ == "__main__":
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.5"
    }

    URL = "https://www.amazon.com/s?k=wall+paint&rh=n%3A21512905011%2Cp_n_feature_three_browse-bin%3A17870886011%2Cp_n_feature_twelve_browse-bin%3A12897841%257C12897931&dc&crid=3E1E8K8OO4Q80&qid=1775412415&rnid=2246753011&sprefix=wall+paint%2Caps%2C625&ref=sr_nr_p_n_feature_twelve_browse-bin_4&ds=v1%3AAMGsfDMBp388HVqSldaamPWsOVLo%2BTw6fOcROOxgcCo"
    webpage = requests.get(URL, headers=HEADERS)
    soup = BeautifulSoup(webpage.content, "html.parser")

    links = soup.find_all("a", attrs={'class':'a-link-normal s-no-outline'})
    links_list = [urljoin("https://www.amazon.com", link.get('href')) for link in links]

    d = {
        "title":[], "price_raw":[], "price_num":[],
        "rating_raw":[], "rating_num":[],
        "reviews_raw":[], "reviews_num":[],
        "paint_type":[], "base_type":[],
        "availability":[], "availability_flag":[],
        "brand":[], "category":[], "url":[], "scrape_date":[]
    }

    # Define paint_type based on the URL search query
    paint_type = "Wall Paint"

    for link in links_list:
        try:
            new_webpage = requests.get(link, headers=HEADERS, timeout=10)
            new_webpage.raise_for_status()
        except requests.RequestException:
            continue

        new_soup = BeautifulSoup(new_webpage.content, "html.parser")

        title = get_title(new_soup)
        price = get_price(new_soup)
        rating = get_rating(new_soup)
        reviews = get_review_count(new_soup)
        availability = get_availability(new_soup)
        base_type = get_base_type(title) # Get base type

        # Numeric conversions
        price_num = safe_float(price)
        rating_num = safe_float(rating.split()[0]) if rating else None
        reviews_num = safe_int(reviews)

        availability_flag = "In Stock" in availability

        d["title"].append(title)
        d["price_raw"].append(price)
        d["price_num"].append(price_num)
        d["rating_raw"].append(rating)
        d["rating_num"].append(rating_num)
        d["reviews_raw"].append(reviews)
        d["reviews_num"].append(reviews_num)
        d["paint_type"].append(paint_type)
        d["base_type"].append(base_type)
        d["availability"].append(availability)
        d["availability_flag"].append(availability_flag)
        d["brand"].append(get_brand(new_soup))
        d["category"].append(get_category(new_soup))
        d["url"].append(link)
        d["scrape_date"].append(datetime.now().strftime("%Y-%m-%d"))

    amazon_df = pd.DataFrame.from_dict(d)
    amazon_df = amazon_df.dropna(subset=["title"]).drop_duplicates(subset=["title"])
    amazon_df.to_csv("amazon_paint_data.csv", header=True, index=False)

In [48]:
amazon_df.head()

,title,price_raw,price_num,rating_raw,rating_num,reviews_raw,reviews_num,paint_type,base_type,availability,availability_flag,brand,category,url,scrape_date
0,THE ONE All-In-One Paint & Primer - Green Matt...,$35.16,35.16,4.1 out of 5 stars,4.1,"(24,254)",24254.0,Wall Paint,Water-Based,In Stock,True,Visit the THE ONE Store,Tools & Home Improvement,https://www.amazon.com/One-Paint-Matte-Litre-U...,2026-04-05
1,"Rust-Oleum 267334 Painter's Touch Latex Paint,...",$18.75,18.75,4.5 out of 5 stars,4.5,"(30,763)",30763.0,Wall Paint,Unknown,In Stock,True,Visit the Rust-Oleum Store,Tools & Home Improvement,https://www.amazon.com/Rust-Oleum-267334-Paint...,2026-04-05
2,THE ONE All-In-One Paint & Primer - Black Sati...,$41.95,41.95,4.1 out of 5 stars,4.1,"(24,254)",24254.0,Wall Paint,Water-Based,In Stock,True,Visit the THE ONE Store,Tools & Home Improvement,https://www.amazon.com/ONE-Paint-Interior-Exte...,2026-04-05
3,"Falling in Art Chalkboard Paint, 16 Oz Black C...",$12.87,12.87,4.6 out of 5 stars,4.6,(70),70.0,Wall Paint,Unknown,In Stock,True,Visit the Falling in Art Store,Tools & Home Improvement,https://aax-us-east-retail-direct.amazon.com/x...,2026-04-05
5,Rust-Oleum 376514 Advanced Dry Door & Trim Pai...,$17.97,17.97,4.6 out of 5 stars,4.6,"(1,820)",1820.0,Wall Paint,Unknown,In Stock,True,Visit the Rust-Oleum Store,Tools & Home Improvement,https://www.amazon.com/Rust-Oleum-376514-Advan...,2026-04-05
